In [3]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from alerce.core import Alerce

def load_curves(csv_dir, min_pts=20, max_pts=250):
    """
    Same preprocessing as in DBSCAN.ipynb.
    """
    curves_data = []  
    files = os.listdir(csv_dir)
    for fname in tqdm(files, desc="Processing curves"):
        oid = fname.replace(".csv", "")
        df = pd.read_csv(os.path.join(csv_dir, fname))
        g = df[df['fid'] == 1]
        r = df[df['fid'] == 2].sort_values('mjd')
        
        if len(r) < min_pts or len(g) < min_pts or len(r) > max_pts:
            continue
        
        color_offset = g['magpsf'].median() - r['magpsf'].median()
        mag = r['magpsf'].values + color_offset
        t = r['mjd'].values
        
        if np.any(np.isnan(mag)) or np.any(np.isnan(t)):
            continue
        
        # Normalize
        t_norm = (t - t.min()) / (t.max() - t.min() + 1e-9)
        mag_norm = (mag - mag.min()) / (mag.max() - mag.min() + 1e-9)
        
        curves_data.append((oid, t_norm, mag_norm, t, mag, df['ra'][0], df['dec'][0]))
    
    return curves_data

In [5]:
csv_dir = "csv_files"

print("=" * 50)
curves_data = load_curves(csv_dir)
print(f"Loaded {len(curves_data)} curves")

Processing curves: 100%|██████████| 12258/12258 [00:34<00:00, 351.55it/s]

Loaded 2869 curves


In [6]:
import numpy as np
from scipy.signal import find_peaks

def is_periodic(t, mag, threshold=0.3):
    """
    Check if a curve is periodic using FFT.
    
    t : np.array
        Time values (assumed normalized 0-1)
    mag : np.array
        Magnitudes (normalized 0-1)
    threshold : float
        Minimum relative peak height to consider periodic
    
    Returns:
        periodic (bool), dominant_period (float)
    """
    N = len(mag)
    
    # Detrend / remove mean
    mag_detrended = mag - np.mean(mag)
    
    # Apply window to reduce spectral leakage
    window = np.hanning(N)
    mag_windowed = mag_detrended * window
    
    # FFT
    X = np.fft.fft(mag_windowed)
    freqs = np.fft.fftfreq(N, d=(t[1]-t[0]))  # sampling interval
    magnitude = np.abs(X)
    
    # Only positive frequencies
    pos_mask = freqs > 0
    freqs = freqs[pos_mask]
    magnitude = magnitude[pos_mask]
    
    # Find peaks
    peaks, props = find_peaks(magnitude, height=threshold * np.max(magnitude))
    
    if len(peaks) == 0:
        return False, None
    
    # Dominant frequency
    dom_freq = freqs[peaks[np.argmax(props['peak_heights'])]]
    dom_period = 1 / dom_freq
    
    return True, dom_period

In [7]:
curves = load_curves(csv_dir)

for oid, t_norm, mag_norm, t, mag, ra, dec in curves:
    periodic, period = is_periodic(t_norm, mag_norm)
    if periodic:
        print(f"{oid} is periodic with period ≈ {period:.3f} (normalized time units)")
    else:
        print(f"{oid} is not periodic")

Processing curves: 100%|██████████| 12258/12258 [00:34<00:00, 351.10it/s]
/apps/external/anaconda/2025.12/lib/python3.13/site-packages/numpy/fft/_helper.py:170: RuntimeWarning: divide by zero encountered in scalar divide
  val = 1.0 / (n * d)
/apps/external/anaconda/2025.12/lib/python3.13/site-packages/numpy/fft/_helper.py:177: RuntimeWarning: invalid value encountered in multiply
  return results * val


ZTF23aanxrjm is periodic with period ≈ 0.078 (normalized time units)
ZTF20abuxrsg is not periodic
ZTF24abmyoit is not periodic
ZTF21abbuzez is periodic with period ≈ 0.023 (normalized time units)
ZTF20abwtifz is not periodic
ZTF18adlwqsr is periodic with period ≈ 0.025 (normalized time units)
ZTF21aaoeuuz is periodic with period ≈ 0.113 (normalized time units)
ZTF20acynqzb is not periodic
ZTF18abgmqhi is not periodic
ZTF21abfxqfi is not periodic
ZTF18abpxwha is periodic with period ≈ 0.002 (normalized time units)
ZTF23aaguxwt is periodic with period ≈ 0.049 (normalized time units)
ZTF18abbmuoo is periodic with period ≈ 0.178 (normalized time units)
ZTF24aacljmq is periodic with period ≈ 3.819 (normalized time units)
ZTF23aaohqcn is not periodic
ZTF18aaccoib is periodic with period ≈ 0.955 (normalized time units)
ZTF20abotkfn is not periodic
ZTF18abtlyqj is periodic with period ≈ 0.044 (normalized time units)
ZTF20aasvqny is periodic with period ≈ 0.635 (normalized time units)
ZTF19abag